# Where Does the Adaptive-Chunking Headroom Live? (Phase B)

Our negative result showed the per-**document** optimal leaf size is not an exploitable function of any document-intrinsic property. This notebook answers the follow-up: **where does the oracle headroom actually live?** It runs the *enriched* leaf-size sweep (now persisting per-question scores) and attributes the gap between a tuned global fixed size and a per-question oracle to named components via **nested oracles**:

| rung | chooses one size per… | interpretation |
|---|---|---|
| **O0** | everything | best global fixed (baseline) |
| **O1** | document | document-intrinsic headroom (≈ predictable-0, from the paper) |
| **O2** | query type (evidence multiplicity `m`) | query-type-attributable headroom |
| **O3** | (document × type) | interaction |
| **O4** | individual question | the absolute ceiling |

**The money quantity is `O4 − O1`** — headroom that varies *within a document across its questions*. A per-document feature is one number per document; it **cannot** produce different sizes for different questions of the same document, so `O4 − O1` is a hard ceiling that makes the H2 negative result inevitable, not incidental.

**SLIDERS axis (arXiv:2604.22294):** `m` = number of gold-evidence paragraphs (1 = lookup, ≥3 = aggregation). Stratifying the decomposition by `m` makes SLIDERS' "aggregation bottleneck" quantitative. **No LLM anywhere.**


In [ ]:
# 1) Code (must be the ckraptor branch WITH the enriched sweep + headroom module).
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

# Sanity: the enriched API must be present, else you cloned an old cache.
from experiments.granularity_sweep import question_meta
from experiments.headroom_decomposition import decompose, cells_from_records, oracle_by_key
print('enriched sweep + headroom_decomposition present ✅')

In [ ]:
# 2) Config + Drive cache. SCALED run = QASPER full test split (N_DOCS=None).
CORPUS = 'qasper'         # 'qasper' (gold-evidence, m is meaningful) | 'quality' (answer-recall, m≡0)
N_DOCS = None             # None = FULL QASPER test split (scaled run). 50 reproduces the paper cohort;
                          # 150–200 is the plan §5 mid-target. Your prior n=50 cache is reused (prefix).
SIZES = [50, 100, 150, 200, 300, 400]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'
import os
os.makedirs(RUN_DIR, exist_ok=True)
# NOTE: a NEW cache filename — old sweeps lack `per_question`, so we must regenerate.
# Same filename across N_DOCS on purpose: run_sweep skips already-done (doc,size) pairs,
# so scaling 50 -> full only computes the NEW docs (the first 50 are read from cache).
OUT = os.path.join(RUN_DIR, f'{CORPUS}_headroom_sweep.json')
SUMMARY = os.path.join(RUN_DIR, f'{CORPUS}_headroom_summary.json')
print('CORPUS=', CORPUS, '| N_DOCS=', N_DOCS, '| OUT=', OUT, '(exists:', os.path.exists(OUT), ')')
try:
    import torch
    print('CUDA:', torch.cuda.is_available(),
          '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
except Exception as e:
    print('torch not ready:', e)

In [ ]:
# 3) Load the corpus. QASPER ships gold evidence -> m = evidence multiplicity is real.
from experiments.datasets import get_loader
import statistics

if CORPUS == 'qasper':
    docs = get_loader('qasper').load(limit=N_DOCS)
elif CORPUS == 'quality':
    # answer-recall proxy path (m will be 0 for all -> query-type axis is degenerate).
    docs = get_loader('quality', split='validation').load(limit=None)
    for d in docs:
        for q in d.questions:
            if q.gold_index is not None and q.options and 0 <= q.gold_index < len(q.options):
                q.gold_answers = [q.options[q.gold_index]]
            else:
                q.gold_answers = []
    docs = [d for d in docs if any((q.gold_answers or []) for q in d.questions)][:N_DOCS]
else:
    raise ValueError(CORPUS)

nq = sum(len(d.questions) for d in docs)
doclens = [len(d.text.split()) for d in docs]
print(f'{len(docs)} {CORPUS} docs; {nq} questions; '
      f'median doc len = {int(statistics.median(doclens))} tokens')

In [ ]:
# 4) Run the ENRICHED sweep (resumable; no LLM). Records now carry `per_question`.
from experiments import granularity_sweep as gs, granularity_calibration as gc
from tqdm.auto import tqdm

coverage_fn = gs._answer_coverage_q if CORPUS == 'quality' else gs._evidence_coverage_q
records = gs.run_sweep(
    docs, SIZES, budget=2000, out_path=OUT,
    coverage_fn=coverage_fn,
    progress=lambda m: tqdm.write(m),
)
print(f'{len(records)} (doc, size) records')

# Guard: if the cache came from an OLD run it has no per_question -> ledger would be empty.
has_pq = any(r.get('per_question') for r in records)
assert has_pq, ('Records lack `per_question` — delete the old OUT cache and re-run so the '
               'enriched sweep regenerates it.')

# H1 sanity: confirm the headroom matches the paper (+9.8% rel. on QASPER n=50).
h = gc.headroom(records)
print(f"H1: best fixed {h['best_fixed_size']} tok @ {h['best_fixed_cov']:.4f} | "
      f"per-doc oracle {h['oracle_cov']:.4f} | +{h['rel_gain_pct']:.1f}% rel.")

In [ ]:
# 5) THE LEDGER. Query type = evidence-multiplicity bucket (1 / 2 / >=3).
def m_bucket(c):
    m = c.get('m', 0)
    return '1' if m <= 1 else ('2' if m == 2 else '>=3')

cells = cells_from_records(records)
print(f'{len(cells)} per-question cells\n')

led = decompose(cells, type_fn=m_bucket)
print(f"{'oracle rung':<28}{'coverage':>10}")
print('-' * 38)
for k in ['O0_global', 'O1_document', 'O2_query_type', 'O3_doc_x_type', 'O4_per_question']:
    print(f'{k:<28}{led[k]:>10.4f}')

c = led['components']
print(f"\n{'headroom component':<28}{'abs':>10}{'% of total':>12}")
print('-' * 50)
tot = c['total'] or 1e-12
for k in ['document', 'query_type', 'interaction', 'within_doc']:
    print(f'{k:<28}{c[k]:>10.4f}{100*c[k]/tot:>11.0f}%')
print(f"{'TOTAL (O4 - O0)':<28}{c['total']:>10.4f}")
print(f"\n>>> MONEY NUMBER  within-doc (O4 - O1) = {led['O4_per_question'] - led['O1_document']:.4f}")
print('    (headroom no per-document feature can EVER capture)')

In [ ]:
# 6) SLIDERS axis: stratify by evidence multiplicity m and decompose within each bucket.
#    D1: document component stays ~0 across m.  D2: total & within-doc grow with m.
#    D3: even O4 (the ceiling) coverage falls as m rises -> aggregation is retrieval-hard.
from collections import Counter

buckets = {'m=1 (lookup)': lambda c: c.get('m', 0) == 1,
           'm=2 (multi-hop)': lambda c: c.get('m', 0) == 2,
           'm>=3 (aggregation)': lambda c: c.get('m', 0) >= 3}

mdist = Counter(cell['m'] for cell in cells)
print('evidence-multiplicity distribution (cells):', dict(sorted(mdist.items())), '\n')

hdr = f"{'stratum':<22}{'#cells':>7}{'O0':>8}{'O1':>8}{'O4':>8}{'doc':>8}{'within':>8}"
print(hdr); print('-' * len(hdr))
per_m = {}
for name, pred in buckets.items():
    sub = [c for c in cells if pred(c)]
    if not sub:
        print(f'{name:<22}{0:>7}   (no cells)'); continue
    l = decompose(sub, type_fn=lambda c: 'const')  # single m -> query_type collapses to 0
    within = l['O4_per_question'] - l['O1_document']
    per_m[name] = {'n_cells': len(sub), 'O0': l['O0_global'], 'O1': l['O1_document'],
                   'O4': l['O4_per_question'], 'document': l['components']['document'],
                   'within_doc': within}
    print(f"{name:<22}{len(sub):>7}{l['O0_global']:>8.3f}{l['O1_document']:>8.3f}"
          f"{l['O4_per_question']:>8.3f}{l['components']['document']:>8.3f}{within:>8.3f}")
print('\nRead: O0->O4 gap widening with m = D2; O4 (abs) falling with m = D3; doc col flat = D1.')

In [ ]:
# 7) HELD-OUT ledger (plan §4.2 / "exactly like H2"): de-bias the oracle optimism.
#    The cell-5 ledger fits each oracle's size on the SAME questions it scores ->
#    optimistic. Here we split the QUESTIONS into fit/score, fit each rung's
#    per-key argmax size on the FIT half, and SCORE every rung on the disjoint
#    half. All five rungs are scored on the SAME held-out set, so they stay
#    comparable and the components telescope. Repeat over N_SPLITS and report
#    mean ± std + a 95% band. O0-O3 become de-biased; O4 is the per-question
#    CEILING (a held-out question was never fit -> it can only be the in-sample
#    max on the score set), so `within_doc = O4 - O3` is a labelled UPPER BOUND.
#    The headline negative result is the DE-BIASED `document` component (O1-O0).
import random, statistics

N_SPLITS = 50        # like H2; raise for smoother bands (pure-Python, no GPU)
FRAC_FIT = 0.7

# Per-question record (once): qk=(doc_id,qid) -> {doc_id, m, sizes:{size:coverage}}
Q = {}
for cell in cells:
    qk = (cell['doc_id'], cell.get('qid'))
    e = Q.setdefault(qk, {'doc_id': cell['doc_id'], 'm': cell.get('m', 0), 'sizes': {}})
    e['sizes'][cell['size']] = cell['coverage']
qks = list(Q)
print(f'{len(qks)} question units; held-out over {N_SPLITS} question-splits '
      f'(fit {int(FRAC_FIT*100)}% / score {100-int(FRAC_FIT*100)}%)\n')

def _mbk(m):
    return '1' if m <= 1 else ('2' if m == 2 else '>=3')
KEYS = {'all':  lambda qk: 'all',
        'doc':  lambda qk: Q[qk]['doc_id'],
        'type': lambda qk: _mbk(Q[qk]['m']),
        'dt':   lambda qk: (Q[qk]['doc_id'], _mbk(Q[qk]['m']))}

def _fit_sizes(fit_qks, group_of):
    grp = {}
    for qk in fit_qks:
        g = group_of(qk)
        gg = grp.setdefault(g, {})
        for s, cov in Q[qk]['sizes'].items():
            gg.setdefault(s, []).append(cov)
    return {g: min(s2c, key=lambda s: (-(sum(s2c[s]) / len(s2c[s])), s))
            for g, s2c in grp.items()}

def _score(score_qks, group_of, chosen, gsize):
    covs = []
    for qk in score_qks:
        s = chosen.get(group_of(qk), gsize)
        sizes = Q[qk]['sizes']
        cov = sizes.get(s, sizes.get(gsize))
        if cov is not None:
            covs.append(cov)
    return sum(covs) / len(covs) if covs else 0.0

metrics = ['O0_global', 'O1_document', 'O2_query_type', 'O3_doc_x_type', 'O4_per_question',
           'document', 'query_type', 'interaction', 'within_doc', 'total', 'money_O4_minus_O1']
acc = {k: [] for k in metrics}
for split in range(N_SPLITS):
    rng = random.Random(1000 + split)
    ids = sorted(qks); rng.shuffle(ids)
    ntr = int(round(FRAC_FIT * len(ids)))
    fit, score = ids[:ntr], ids[ntr:]
    if not fit or not score:
        continue
    gsize = _fit_sizes(fit, KEYS['all'])['all']
    o0 = _score(score, KEYS['all'],  {'all': gsize}, gsize)
    o1 = _score(score, KEYS['doc'],  _fit_sizes(fit, KEYS['doc']),  gsize)
    o2 = _score(score, KEYS['type'], _fit_sizes(fit, KEYS['type']), gsize)
    o3 = _score(score, KEYS['dt'],   _fit_sizes(fit, KEYS['dt']),   gsize)
    o4 = statistics.fmean(max(Q[qk]['sizes'].values()) for qk in score)  # in-sample ceiling
    vals = {'O0_global': o0, 'O1_document': o1, 'O2_query_type': o2,
            'O3_doc_x_type': o3, 'O4_per_question': o4,
            'document': o1 - o0, 'query_type': o2 - o0,
            'interaction': o3 - o1 - o2 + o0, 'within_doc': o4 - o3,
            'total': o4 - o0, 'money_O4_minus_O1': o4 - o1}
    for k, v in vals.items():
        acc[k].append(v)

def _band(xs):
    xs = sorted(xs)
    lo = xs[max(0, int(round(0.025 * (len(xs) - 1))))]
    hi = xs[min(len(xs) - 1, int(round(0.975 * (len(xs) - 1))))]
    return lo, hi

heldout = {}
print(f"{'quantity':<20}{'mean':>8}{'std':>8}{'   95% band':>18}")
print('-' * 54)
for k in metrics:
    xs = acc[k]
    mu, sd = statistics.fmean(xs), (statistics.pstdev(xs) if len(xs) > 1 else 0.0)
    lo, hi = _band(xs)
    heldout[k] = {'mean': mu, 'std': sd, 'lo': lo, 'hi': hi}
    star = ' *' if lo > 0 else ('  0' if lo <= 0 <= hi else '')
    print(f'{k:<20}{mu:>8.4f}{sd:>8.4f}   [{lo:>7.4f}, {hi:>7.4f}]{star}')
print('\n* = 95% band excludes 0 ;  "0" = band includes 0.')
print('KEY: de-biased `document` (O1-O0) near 0 with a band spanning 0 == the H2')
print('negative result, now rigorous. `within_doc` is an UPPER BOUND (O4 is the')
print('in-sample per-question ceiling). Compare against the cell-5 in-sample ledger:')
print('held-out `document` should be SMALLER (optimism removed).')

In [ ]:
# 8) Persist a summary for the writeup: in-sample point ledger + held-out CIs.
import json
summary = {
    'corpus': CORPUS, 'n_docs': len(docs), 'n_questions': nq,
    'n_cells': len(cells), 'n_question_units': len(qks),
    'headroom_H1': h,
    # In-sample point ledger (cell 5) — interpretable, additive, but optimistic.
    'ledger_insample': {k: led[k] for k in ['O0_global', 'O1_document', 'O2_query_type',
                                            'O3_doc_x_type', 'O4_per_question']},
    'components_insample': led['components'],
    'within_doc_O4_minus_O1_insample': led['O4_per_question'] - led['O1_document'],
    # Held-out, de-biased ledger (cell 7): the paper numbers. {metric: {mean,std,lo,hi}}.
    'heldout': heldout,
    'n_splits': N_SPLITS, 'frac_fit': FRAC_FIT,
    'per_m_stratum': per_m,
    'm_distribution': {str(k): v for k, v in sorted(mdist.items())},
}
with open(SUMMARY, 'w') as fh:
    json.dump(summary, fh, indent=2, default=float)
print('wrote', SUMMARY)
print(json.dumps(summary, indent=2, default=float))

## How to read this

1. **H1 sanity (cell 4).** At **full scale the headroom % will differ from the n=50 paper number (+9.8% rel.)** — that is expected, and the *new* number is the one to report. The cell-4 `assert` only checks that `per_question` is present, so a different value will not fail it. Only worry if it prints an *empty/None* ledger downstream.
2. **In-sample ledger (cell 5).** The four components sum to the total (O4−O0) by construction. This view is *interpretable but optimistic* (each oracle picks a size using the same questions it is scored on). Use it to see the shape; use **cell 7** for the numbers you report.
3. **SLIDERS axis (cell 6).** If total/within-doc headroom **grows with `m`** while the document component stays ~0, and O4's absolute coverage **falls** at `m≥3`, that empirically motivates the switch from retrieval-granularity tuning to SLIDERS-style structured extraction for aggregation questions.
4. **Held-out ledger (cell 7) — THE PAPER NUMBERS.** Sizes are fit on 70% of questions and scored on the disjoint 30%, over 50 splits (mean ± std + 95% band), *exactly like H2*. This removes the in-sample optimism. Read it as:
   - **`document` (O1−O0)** de-biased and **near 0 with a band spanning 0** ⇒ the H2 negative result, now rigorous. It should be *smaller* than the cell-5 in-sample `document` (that gap **is** the optimism you removed).
   - **`within_doc` (O4−O3)** is an **upper bound** (O4 is the per-question ceiling, which can't be held out). A large `within_doc` with its band clear of 0 = the headroom lives within-document, unreachable by any per-document feature.
   - A `*` marks a quantity whose 95% band excludes 0.
5. Paste the printed summary (`*_headroom_summary.json` now carries `heldout` + `components_insample`) back to fold into `paper/main.tex` and a `docs/results/` writeup.
